# Install lib


In [ ]:
!pip install wandb
!pip install transformers
!pip install trl
!pip install datasets
!pip install torch
!pip install accelerate
!pip install bitsandbytes
!pip install peft
!pip install numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.0/348.0 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 128.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Orpo

In [ ]:
# === SETUP ===
import os
import time
import json
import torch
import wandb
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import ORPOConfig, ORPOTrainer
from transformers import EarlyStoppingCallback

In [ ]:
# 🚀 Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ✅ Suppress warnings
os.environ["WANDB_DISABLE_INTERNAL_MESSAGES"] = "true"
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = 'true'

# ✅ Initialize WandB
wandb.login()
wandb.init(
    project="ORPO",
    name="epoch1_general",
    config={"beta": 0.1, "learning_rate": 2e-5, "batch_size": 8},
)

# ✅ Set dataset paths (General Info)
base_path = "/content/drive/My Drive/ORPO"
train_path = f"{base_path}/train_gen.json"
test_path = f"{base_path}/test_gen.json"

# ✅ Load JSON dataset
with open(train_path, "r", encoding="utf-8") as f:
    train_data = json.load(f)
with open(test_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

# ✅ Process the dataset
def process_data(data):
    return [
        {"prompt": e["prompt"].strip(), "chosen": e["chosen"].strip(), "rejected": e["rejected"].strip()}
        for e in data if all(k in e and e[k].strip() for k in ["prompt", "chosen", "rejected"])
    ]

train_data_processed = process_data(train_data)
test_data_processed = process_data(test_data)

train_dataset = Dataset.from_list(train_data_processed)
test_dataset = Dataset.from_list(test_data_processed)

dataset = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})

# ✅ Load SeaLLMs-v3-7B-Chat **without quantization**
model_name = "SeaLLMs/SeaLLMs-v3-1.5B-Chat"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, padding_side="right")
model = AutoModelForCausalLM.from_pretrained(model_name).to("cuda")

# 🚀 Enable Memory Optimization (Gradient Checkpointing)
model.gradient_checkpointing_enable()

# ✅ ORPO Training Config (Optimized for 40GB GPU)
orpo_config = ORPOConfig(
    output_dir=f"{base_path}/Output/ORPO/Epoch_1",
    num_train_epochs=1,  # First Epoch
    per_device_train_batch_size=8,  # High batch size (40GB GPU can handle this)
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,  # Effective batch size = 32 (8 * 4)
    learning_rate=2e-5,
    weight_decay=0.01,
    max_grad_norm=1.0,
    warmup_ratio=0.06,
    lr_scheduler_type="cosine",
    save_steps=500,
    logging_steps=100,
    evaluation_strategy="steps",
    eval_steps=500,
    report_to=["wandb"],
    beta=0.1,
    fp16=True,  # Mixed precision for faster training
)

# ✅ Tokenization function
max_length = 2048  # Full context length for SeaLLMs
def tokenize_function(examples):
    prompt_inputs = tokenizer(examples["prompt"], truncation=True, padding="max_length", max_length=max_length)
    chosen_inputs = tokenizer(examples["chosen"], truncation=True, padding="max_length", max_length=max_length)
    rejected_inputs = tokenizer(examples["rejected"], truncation=True, padding="max_length", max_length=max_length)

    return {
        "prompt": examples["prompt"],
        "chosen": examples["chosen"],
        "rejected": examples["rejected"],
        "prompt_input_ids": prompt_inputs["input_ids"],
        "prompt_attention_mask": prompt_inputs["attention_mask"],
        "chosen_input_ids": chosen_inputs["input_ids"],
        "chosen_attention_mask": chosen_inputs["attention_mask"],
        "rejected_input_ids": rejected_inputs["input_ids"],
        "rejected_attention_mask": rejected_inputs["attention_mask"],
    }

# ✅ Tokenize dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=dataset["train"].column_names)

# ✅ Initialize ORPO Trainer
orpo_trainer = ORPOTrainer(
    model=model,
    args=orpo_config,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
    tokenizer=tokenizer
)

# 🚀 Start Training
orpo_trainer.train()

# ✅ Evaluate the trained model
eval_prompt = "Paano nakakatulong ang basil extract sa pag-kontrol ng aphid sa mais? / How does basil extract help manage aphid infestations in corn?"
model_input = tokenizer(eval_prompt, return_tensors="pt", padding=True, truncation=True, max_length=768)
attention_mask = model_input['attention_mask']

# 🚀 Generate response
model.eval()
with torch.no_grad():
    output = model.generate(model_input["input_ids"].to("cuda"), attention_mask=attention_mask, max_length=150)
    print(tokenizer.decode(output[0], skip_special_tokens=True))

# ✅ Log training time
training_time = time.time() - start_time
print(f"Training took {training_time:.2f} seconds")
wandb.log({"training_time_seconds": training_time})

# ✅ Save fine-tuned model
model.save_pretrained(f"{base_path}/Trained_Model/ORPO/epoch1")
tokenizer.save_pretrained(f"{base_path}/Trained_Model/ORPO/epoch1")

# ✅ Finish WandB
wandb.finish()

Mounted at /content/drive


<IPython.core.display.Javascript object>

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nickolaschase-ling-eng (nickolaschase) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


No validation set found. Proceeding with train and test only.


/usr/local/lib/python3.11/dist-packages/transformers/models/auto/tokenization_auto.py:898: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


...

Map:   0%|          | 0/9528 [00:00<?, ? examples/s]

Map:   0%|          | 0/9528 [00:00<?, ? examples/s]

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Step,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 64.00 MiB. GPU 0 has a total capacity of 22.16 GiB of which 47.38 MiB is free. Process 9904 has 22.11 GiB memory in use. Of the allocated memory 21.71 GiB is allocated by PyTorch, and 165.16 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
# Mount Google Drive to access datasets and save models
from google.colab import drive
drive.mount('/content/drive')

# Load the fine-tuned model
base_path = "/content/drive/My Drive/ORPO"
model_path = f"{base_path}/finetuned1_epoch5"

# Load the model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Evaluation function
def evaluate_prompts(prompts):
    model.eval()
    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model.generate(
                inputs["input_ids"],
                max_length=150,
                num_return_sequences=1,
                temperature=0.7,
                top_p=0.9
            )
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        print(f"Prompt: {prompt}\nResponse: {generated_text}\n{'-'*60}")

# Unseen evaluation prompts
unseen_prompts = [
    "What pests affect cassava?",
    "Paano iwasan ang peste sa kamoteng kahoy?",
    "How to control aphids in corn?",
    "Anong mga peste ang karaniwang sumisira sa saging?",
    "What are the symptoms of pest infestation in coconut trees?",
    "Paano makakatulong ang neem oil sa pag-iwas ng peste sa mangga?"
]

# Run evaluation
evaluate_prompts(unseen_prompts)


Mounted at /content/drive


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Prompt: What pests affect cassava?
Response: What pests affect cassava?Cassava is affected by many pests, including whiteflies, mealybugs, termites, and cassava hornworms. Whiteflies suck plant sap and can cause yellowing of leaves. Mealybugs produce sticky honeydew, which leads to fungal infections like sooty mold. Termites weaken the roots and attack cassava stems. If left unmanaged, these pests can significantly reduce cassava yields.
------------------------------------------------------------
Prompt: Paano iwasan ang peste sa kamoteng kahoy?
Response: Paano iwasan ang peste sa kamoteng kahoy?Narito ang mga hakbang upang maiwasan ang peste sa kamoteng kahoy:

1. Gumamit ng malulusog at walang sakit na mga punla – Siguraduhing ang mga seedlings ay malusog at walang peste bago itanim upang maiwasan ang maagang pinsala.
2. Maglagay ng organikong panlaban sa peste – Magpakawala ng mga ladybugs, putakti, at gagamba na kumakain ng maliliit na peste tulad ng aphids at mealybugs.
3. Mag-s


In [ ]:
import os

# This will disconnect and delete the runtime in Google Colab
os.kill(os.getpid(), 9)